# Positional Encoding in Transformers

Below are **clean, self-contained Python/NumPy examples** showing:

1. **Sinusoidal Positional Encoding**
2. **Learned Absolute Positional Embedding (NumPy version for illustration)**
3. **Rotary Positional Embedding (RoPE)**
4. **ALiBi bias computation**

Everything is written *framework-free* (pure NumPy, except where noted).

# ✅ 1. Sinusoidal Positional Encoding (Vaswani et al.)

In [1]:
import numpy as np

def sinusoidal_positional_encoding(max_len, d_model):
    """
    Returns PE of shape (max_len, d_model)
    """
    pe = np.zeros((max_len, d_model))

    position = np.arange(max_len)[:, None]  # shape (max_len, 1)
    div_term = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))  # shape (d_model/2)

    # even dimensions = sin
    pe[:, 0::2] = np.sin(position * div_term)
    # odd dimensions = cos
    pe[:, 1::2] = np.cos(position * div_term)

    return pe


# Example
pe = sinusoidal_positional_encoding(10, 16)
print(pe.shape)
print(pe[0])      # position 0
print(pe[5])      # position 5

(10, 16)
[0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1. 0. 1.]
[-0.95892427  0.28366219  0.99994652 -0.01034232  0.47942554  0.87758256
  0.1574559   0.98752602  0.04997917  0.99875026  0.01581073  0.999875
  0.00499998  0.9999875   0.00158114  0.99999875]


✔ Multi-frequency\
✔ Deterministic\
✔ Extrapolates to unseen sequence lengths

# ✅ 2. Learned Absolute Positional Embeddings (simple NumPy toy version)

Even though in practice these are learned by backprop (e.g., PyTorch), here is the structural logic:

In [5]:
import numpy as np

class LearnedAbsolutePE:
    def __init__(self, max_len, d_model):
        self.P = np.random.randn(max_len, d_model) * 0.01  # trainable in real models

    def __call__(self, positions):
        """
        positions: array of indices (e.g., [0, 1, 2, 3])
        """
        return self.P[positions]  # lookup


# Example
pe_layer = LearnedAbsolutePE(max_len=10, d_model=16)

positions = np.arange(10)
pe = pe_layer(positions)
print(f'PE shape: {pe.shape}')
print(f'PE for positions {positions}:')

PE shape: (10, 16)
PE for positions [0 1 2 3 4 5 6 7 8 9]:


✔ Learns arbitrary patterns\
✘ Cannot extrapolate beyond `max_len`

# ✅ 3. RoPE – **Rotary Positional Encoding** (NumPy)

### Key idea

For each 2D pair `(x_2k, x_2k+1)` apply rotation by angle
$$
\theta_k \cdot \text{position}
$$

In [3]:
import numpy as np

def build_rope_frequencies(d_model, base=10000):
    """
    Returns the frequencies (theta) for RoPE.
    """
    half_dim = d_model // 2
    freq_exponents = np.arange(half_dim) / half_dim
    return 1.0 / (base ** freq_exponents)  # shape (half_dim,)


def apply_rope(x, pos, freqs):
    """
    x: (..., d_model) input
    pos: integer position
    freqs: frequencies array of shape (d_model/2)
    """
    d_model = x.shape[-1]
    half = d_model // 2

    x1 = x[..., :half]
    x2 = x[..., half:]

    # rotation angles
    angles = pos * freqs  # shape (half,)

    sin = np.sin(angles)
    cos = np.cos(angles)

    # apply rotation per frequency
    x_rotated_1 = x1 * cos - x2 * sin
    x_rotated_2 = x1 * sin + x2 * cos

    return np.concatenate([x_rotated_1, x_rotated_2], axis=-1)


# Example
d_model = 8
freqs = build_rope_frequencies(d_model)

x = np.random.randn(d_model)
out = apply_rope(x, pos=5, freqs=freqs)
print(out)

[-0.3091689  -0.7909491   1.15484433  1.29492399 -0.12406965  0.28532202
 -2.432808    0.34091216]


✔ Encodes **relative** position information\
✔ Great generalization to longer sequences\
✔ Used in LLaMA, GPT-NeoX, PaLM variants

# ✅ 4. ALiBi (Attention Linear Biases)

ALiBi adds a **per-head slope × distance** bias to attention logits:

$$
\text{bias}(i,j) = -\text{slope}_h \cdot (j - i)
$$

In [4]:
import numpy as np

def build_alibi_slopes(num_heads):
    """
    Slopes recommended from the ALiBi paper.
    """
    def get_slopes(n):
        # recursively defined in original implementation
        import math
        if math.log2(n).is_integer():
            start = 2**(-2**-(math.log2(n)-3))
            return [start * (2**i) for i in range(n)]
        else:
            # find power of 2
            closest_power_of_2 = 2 ** int(np.log2(n))
            return get_slopes(closest_power_of_2) + build_alibi_slopes(n - closest_power_of_2)

    return np.array(get_slopes(num_heads))


def alibi_bias(seq_len, slopes):
    """
    Returns bias tensor of shape (num_heads, seq_len, seq_len)
    """
    num_heads = len(slopes)
    i = np.arange(seq_len)[:, None]  # shape (seq_len, 1)
    j = np.arange(seq_len)[None, :]  # shape (1, seq_len)

    dist = j - i  # positive if j is to the right

    # bias[h, i, j] = -slopes[h] * dist[i, j]
    return -slopes[:, None, None] * dist[None, :, :]


# Example
slopes = build_alibi_slopes(4)
bias = alibi_bias(seq_len=5, slopes=slopes)
print(bias.shape)  # (4 heads, 5, 5)

(4, 5, 5)


✔ Strong extrapolation to long sequences\
✔ Simple to implement\
✘ Only monotonic distance penalty